In [ ]:
# importing dependencies
import pandas as pd # data frame
import numpy as np # matrix math
from scipy.io import wavfile # reading the wavfile
import os # interation with the OS
from sklearn.utils import shuffle # shuffling of data
from random import sample # random selection
from tqdm import tqdm # progress bar
import matplotlib.pyplot as plt # to view graphs
import wave
from math import log, floor
# audio processing
from scipy import signal # audio processing
from scipy.fftpack import dct
import librosa # library for audio processing
import xgboost as xgb
import lightgbm as lgb
import catboost as ctb
from keras.utils import *
from sklearn.ensemble import *
import pickle

In [ ]:
X_train = np.load('feature/mfcc/X_train.npy')
X_train2 = np.load('feature/fbank/X_train.npy')
print(X_train.shape,X_train2.shape)

In [ ]:
SAMPLE_RATE = 44100
fname = 'data/audio_train/002d256b.wav'    # Hi-hat
wav, _ = librosa.core.load(fname, sr=SAMPLE_RATE)
# wav = wav[:2*44100]
# mfcc = librosa.feature.mfcc(wav, sr = SAMPLE_RATE, n_mfcc=400)
wav.shape
#???

In [ ]:
Y_train = pd.read_csv("data/train_label.csv")
labels = list(Y_train['label'].unique())
labels

In [ ]:
dicts={}
for i,v in enumerate(labels):
    dicts[v]=i
Y_train['trans'] = Y_train['label'].map(dicts)
Y_train

In [ ]:
# pickle.dump(dicts,open('data/map.pkl','wb'))
# pickle.dump(dicts_ch,open('data/map_reverse.pkl','wb'))

In [ ]:
# pickle.load(open('data/map_reverse.pkl','rb'))

In [ ]:
dicts_ch={}
for i, v in dicts.items():
    dicts_ch[v]=i
dicts_ch

In [ ]:
Y_train['onehot'] = Y_train['trans'].apply(lambda x: to_categorical(x,num_classes=len(labels)))
Y_train

In [ ]:
train_Y = np.array(Y_train['onehot'].tolist())
train_Y.shape

In [ ]:
xgb_mfcc = xgb.XGBClassifier(tree_method='gpu_hist',predictor='gpu_predictor') #n_jobs=11,
train_X = X_train.reshape(X_train.shape[0],-1)
# xgb_mfcc.fit(train_X,Y_train['trans'].values)

In [ ]:
lgb_fbank = lgb.LGBMClassifier(n_jobs=11,boosting_type='gbdt',max_depth=-1,is_unbalance=True,num_class=41,metric='acc') #,device='gpu' n_jobs=11,
train_X2 = X_train2.reshape(X_train2.shape[0],-1) #n_jobs=-1,
# train_X = X_train.reshape(X_train.shape[0],-1)
lgb_fbank.fit(train_X2,Y_train['trans'].values)

In [ ]:
vc = VotingClassifier(estimators=[('xgb',xgb_mfcc),('lgb',lgb_fbank)],n_jobs=11,voting='soft',flatten_transform=True)
vc.fit(train_X2,Y_train['trans'].values) #n_jobs=2

In [ ]:
test_Y = np.load('feature/fbank/X_test.npy')
# test_Y.shape
Y_test = test_Y.reshape(9400,-1)

In [ ]:
ans = lgb_fbank.predict(Y_test)

In [ ]:
np.save('result/lgb_naive.npy',ans)

In [ ]:
pickle.dump(lgb_fbank,open('model/lgb_naive.pkl','wb'))

In [ ]:
# kk = pickle.load(open('model/lgb_vc.pkl','rb'))
# kk.predict(Y_test)

In [ ]:
up = pd.read_csv('data/sample_submission.csv')
up

In [ ]:
up['ans1'] = pd.Series(ans)
up['trans']=up['ans1'].map(dicts_ch)
up

In [ ]:
# def modify(string):

In [ ]:
# up.label.str.replace('Laughter',up.trans)
up['label'] = up['trans']
up[['fname','label']].to_csv("result/naive_fbank_lgb_submission.csv", index=False)